In a RAG (Retrieval-Augmented Generation) system, data chunking is crucial because LLMs and embedding models have input token limits. Chunking ensures information is split meaningfully to preserve context and semantics while remaining within token constraints.

In [1]:
! pip install langchain tiktoken spacy
! python -m spacy download en_core_web_sm

  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached cymem-2.0.11-cp312-cp312-win_amd64.whl.metadata (8.8 kB)
  Using cached thinc-8.3.6-cp312-cp312-win_amd64.whl.metadata (15 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached srsly-2.5.1-cp312-cp312-win_amd64.whl.metadata (20 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-0.4.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached langcodes-3.5.0-py3-none-any.whl.metadata (29 kB)
  Using cached language_data-1.3.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached blis-1.3.0-cp312-cp312-win_amd64.whl.metadata (7.6 kB)
  Using cached confection-0.1.5-py3-none-any.whl.metadata (19 kB)
   ---------------------------------------- 0.0/13.9 MB ? eta -:--:--
    --------------------------------------- 0.3/13.9 MB ? eta -:--:--
   -- -------------------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.3.2 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.3.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.2 which is incompatible.
scipy 1.13.1 requires numpy<2.3,>=1.22.4, but you have numpy 2.3.2 which is incompatible.
c:\Personal\2024\Learning\Generative AI\RAG\.venv\Scripts\python.exe: No module named spacy


🔹 1. Fixed-Size Chunking
Split text into equal-sized chunks, typically by characters, words, or tokens.

Character-based: Split every N characters

Word-based: Split every N words

Token-based: Split every N tokens (preferred when using tokenizer-aware models)

📌 Simple but can split sentences mid-way and hurt semantic coherence.

✅ Example: Fixed-Size Chunking (Character-Based)

In [2]:
for i in range(0,10,2):
    print(i)


0
2
4
6
8


In [3]:
text = "This is a simple example to demonstrate fixed size chunking. We split the text into chunks of equal length."
len(text)

107

In [4]:
for i in range(0,len(text),50):
    print(i)

0
50
100


In [ ]:
text[50:50+50]

'This is a simple example to demonstrate fixed size'

In [2]:
def fixed_size_chunking(text, chunk_size=50):
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
    return chunks

# Sample text
text = "This is a simple example to demonstrate fixed size chunking. We split the text into chunks of equal length."

# Call the function
chunks = fixed_size_chunking(text, chunk_size=50)

# Display results
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk)
    print("-" * 40)


Chunk 1:
This is a simple example to demonstrate fixed size
----------------------------------------
Chunk 2:
 chunking. We split the text into chunks of equal 
----------------------------------------
Chunk 3:
length.
----------------------------------------


✅ Example 1: Fixed-Size Chunking (Word-Based)

In [3]:
def fixed_word_chunking(text, chunk_size=10):
    words = text.split()
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

# Sample text
text = "This is a simple example to demonstrate fixed size chunking. We split the text into chunks of equal length."

# Call the function
chunks = fixed_word_chunking(text, chunk_size=10)

# Display results
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk)
    print("-" * 40)


Chunk 1:
This is a simple example to demonstrate fixed size chunking.
----------------------------------------
Chunk 2:
We split the text into chunks of equal length.
----------------------------------------


✅ Example 2: Fixed-Size Chunking (Token-Based with tiktoken)

In [4]:
import tiktoken

def fixed_token_chunking(text, chunk_size=10):
    enc = tiktoken.get_encoding("cl100k_base")  # Use encoding for OpenAI models
    tokens = enc.encode(text)
    chunks = [tokens[i:i+chunk_size] for i in range(0, len(tokens), chunk_size)]
    return [enc.decode(chunk) for chunk in chunks]

# Sample text
text = "This is a simple example to demonstrate fixed size chunking. We split the text into chunks of equal length."

# Call the function
chunks = fixed_token_chunking(text, chunk_size=10)

# Display results
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk)
    print("-" * 40)


Chunk 1:
This is a simple example to demonstrate fixed size chunk
----------------------------------------
Chunk 2:
ing. We split the text into chunks of equal
----------------------------------------
Chunk 3:
 length.
----------------------------------------


🔹 2. Sliding Window Chunking
Chunks overlap by a certain percentage or number of tokens.

Purpose: Preserves context between chunks

Parameters:

chunk_size: e.g., 512 tokens

overlap: e.g., 50–100 tokens

📌 Great for minimizing context loss during splitting.

✅ Example: Sliding Window Chunking (Word-Based)

In [5]:
def sliding_window_chunking(text, chunk_size=10, overlap=3):
    words = text.split()
    step = chunk_size - overlap
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), step)]
    return chunks

# Sample text
text = "This is a simple example to demonstrate sliding window chunking. It helps preserve context between chunks by overlapping."

# Call the function
chunks = sliding_window_chunking(text, chunk_size=10, overlap=3)

# Display results
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk)
    print("-" * 40)


Chunk 1:
This is a simple example to demonstrate sliding window chunking.
----------------------------------------
Chunk 2:
sliding window chunking. It helps preserve context between chunks by
----------------------------------------
Chunk 3:
between chunks by overlapping.
----------------------------------------


In [6]:
# Sample text
text = "This is a simple example to demonstrate sliding window chunking. It helps preserve context between chunks by overlapping."

In [7]:
words = text.split()
print(words)

['This', 'is', 'a', 'simple', 'example', 'to', 'demonstrate', 'sliding', 'window', 'chunking.', 'It', 'helps', 'preserve', 'context', 'between', 'chunks', 'by', 'overlapping.']


In [8]:
chunk_size=10
overlap=3
step = chunk_size - overlap
print(step)

7


In [9]:
len(words)

18

In [10]:
for i in range(0, len(words), step):
    print(i)

0
7
14


In [11]:
" ".join(words[i:i+chunk_size])

'between chunks by overlapping.'

In [15]:
chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), step)]
print(chunks)

['This is a simple example to demonstrate sliding window chunking.', 'sliding window chunking. It helps preserve context between chunks by', 'between chunks by overlapping.']


🔹 3. Recursive Text Splitting (LangChain-style)
Intelligently splits text using a recursive hierarchy of separators.

Hierarchy example:

Paragraph → Sentence → Word → Character

Used by: LangChain's RecursiveCharacterTextSplitter

📌 Preserves as much semantic context as possible while fitting token limits.

In [12]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Sample long text
text = """
LangChain is a powerful framework for building applications with language models. 
It provides abstractions and utilities to make LLM-powered apps easier to develop.

This includes support for memory, chains, agents, and retrieval-augmented generation (RAG). 
It also helps with prompt management and evaluation.

LangChain is modular and supports many backends, including OpenAI, Cohere, and HuggingFace.
"""

# Initialize RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,        # maximum size of each chunk (in characters)
    chunk_overlap=20,      # number of characters to overlap
    separators=["\n\n", "\n", ".", " "]  # hierarchy of splitting
)

# Split the text
chunks = text_splitter.split_text(text)

# Display the chunks
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:\n{chunk}\n{'-' * 40}")


Chunk 1:
LangChain is a powerful framework for building applications with language models.
----------------------------------------
Chunk 2:
It provides abstractions and utilities to make LLM-powered apps easier to develop.
----------------------------------------
Chunk 3:
This includes support for memory, chains, agents, and retrieval-augmented generation (RAG).
----------------------------------------
Chunk 4:
It also helps with prompt management and evaluation.
----------------------------------------
Chunk 5:
LangChain is modular and supports many backends, including OpenAI, Cohere, and HuggingFace.
----------------------------------------


🔍 What are separators?
The separators list tells LangChain how to try breaking the text — it recursively falls back through each separator in order until it produces chunks under the given chunk_size.

🧩 Separator List: ["\n\n", "\n", ".", " "]
Each separator serves a specific purpose:

1. \n\n – Paragraph Breaks
Looks for double newlines, usually meaning a paragraph boundary.

Ideal for clean, top-level separation.

2. \n – Line Breaks / Sentences (in structured text)
Useful for breaking up lines, like in transcripts or list-like formats.

A bit finer-grained than \n\n.

3. . – Sentence Endings
A natural fallback when paragraphs/lines are too big.

Keeps full sentences together, rather than splitting mid-sentence.

4. " " – Spaces Between Words
Last resort — splits text at word boundaries.

Used only if larger chunks still exceed the limit.

🔁 How Recursive Splitting Works
Let’s say you set:

chunk_size=100

chunk_overlap=20

separators=["\n\n", "\n", ".", " "]

Here's the step-by-step logic:

Try to split on \n\n. If all chunks are ≤ 100 characters → done!

If any chunk is still >100, take just that chunk and:

Try splitting it using \n

Still too big? Try splitting with .

Still too big? Try " " (spaces)

If all fails, split arbitrarily



🔹 4. Semantic Chunking
Use NLP techniques to split based on meaning or topics.

Methods:

Sentence segmentation + grouping semantically related sentences

Use of models like BERTScore or TextTiling to detect topic shifts

📌 Improves retrieval by keeping each chunk topically cohesive.

🔹 5. Heading-Based or Structural Chunking
Use document structure like:

Headings (H1/H2/H3 in HTML/Markdown)

Sections in PDFs or Word documents

XML/JSON tag hierarchy

📌 Ideal for structured documents such as manuals, academic papers, websites.

In [13]:
from langchain.text_splitter import MarkdownHeaderTextSplitter

markdown_text = """
# Introduction
This is an intro section.

## Details
Detailed info goes here.

# Conclusion
This is the end.
"""

text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "Header1"), ("##", "Header2")])
docs = text_splitter.split_text(markdown_text)
print(docs)

# Each doc contains .page_content and .metadata


[Document(metadata={'Header1': 'Introduction'}, page_content='This is an intro section.'), Document(metadata={'Header1': 'Introduction', 'Header2': 'Details'}, page_content='Detailed info goes here.'), Document(metadata={'Header1': 'Conclusion'}, page_content='This is the end.')]


🔹 6. Sentence-Based Chunking
Split documents into sentences and group a fixed number of sentences per chunk.

📌 Balances structure and readability. Great for QA use cases.

🔹 7. Windowed Sentence Chunking
Similar to sliding window but with sentences.

Move forward by 1–2 sentences at a time while grouping 3–4 sentences per chunk.

📌 Maintains sentence boundaries and context better than plain fixed chunking.

In [18]:
import re

# Sample text
text = """
LangChain is a powerful framework for building applications with language models.
It provides abstractions to help developers build with LLMs.
The framework supports retrieval-augmented generation and agent-based workflows.
LangChain also offers integrations with OpenAI, Cohere, and HuggingFace.
It is modular and flexible for a variety of use cases.
This makes LangChain suitable for production-grade LLM applications.
"""

# Step 1: Manual sentence tokenization using regex
# This pattern looks for periods, question marks, or exclamation marks 
# followed by whitespace or end of string
sentences = re.split(r'(?<=[.!?])\s+', text.strip())
# Remove any empty sentences that might result from the split
sentences = [s for s in sentences if s.strip()]

# Step 2: Windowed chunking
chunk_size = 3  # number of sentences per chunk
overlap = 2     # how many sentences to overlap

chunks = []
for i in range(0, len(sentences), chunk_size - overlap):
    # Make sure we don't go beyond array bounds
    end_idx = min(i + chunk_size, len(sentences))
    window = sentences[i:end_idx]
    if window:
        chunks.append(" ".join(window))
    # Stop if we've processed all sentences
    if end_idx >= len(sentences):
        break

# Step 3: Print results
for i, chunk in enumerate(chunks):
    print(f"🔹 Chunk {i+1}:\n{chunk}\n{'-' * 60}")

🔹 Chunk 1:
LangChain is a powerful framework for building applications with language models. It provides abstractions to help developers build with LLMs. The framework supports retrieval-augmented generation and agent-based workflows.
------------------------------------------------------------
🔹 Chunk 2:
It provides abstractions to help developers build with LLMs. The framework supports retrieval-augmented generation and agent-based workflows. LangChain also offers integrations with OpenAI, Cohere, and HuggingFace.
------------------------------------------------------------
🔹 Chunk 3:
The framework supports retrieval-augmented generation and agent-based workflows. LangChain also offers integrations with OpenAI, Cohere, and HuggingFace. It is modular and flexible for a variety of use cases.
------------------------------------------------------------
🔹 Chunk 4:
LangChain also offers integrations with OpenAI, Cohere, and HuggingFace. It is modular and flexible for a variety of use case

🔹 8. Metadata-Aware Chunking
Preserve metadata like:

In RAG systems, metadata-aware chunking means you retain metadata (like source, page number, document type, etc.) along with each chunk of text. This helps in:

Better filtering during retrieval

Providing source attribution in answers

Organizing large documents by context

✅ Example Setup
Let's say we have two text documents with metadata like "source" and "page".

We’ll use RecursiveCharacterTextSplitter to split the text, and attach metadata to each chunk.

In [19]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# Sample documents with metadata
docs = [
    Document(
        page_content="LangChain is a framework for building apps with LLMs. It supports chains and agents.",
        metadata={"source": "langchain_intro.pdf", "page": 1}
    ),
    Document(
        page_content="Retrieval-Augmented Generation improves accuracy. LangChain supports RAG pipelines.",
        metadata={"source": "rag_guide.pdf", "page": 3}
    )
]

# Initialize a text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)

# Split the documents and keep metadata
chunked_docs = text_splitter.split_documents(docs)

# Print result
for i, doc in enumerate(chunked_docs):
    print(f"🔹 Chunk {i+1}")
    print("Text:", doc.page_content)
    print("Metadata:", doc.metadata)
    print("-" * 60)


🔹 Chunk 1
Text: LangChain is a framework for building apps with
Metadata: {'source': 'langchain_intro.pdf', 'page': 1}
------------------------------------------------------------
🔹 Chunk 2
Text: apps with LLMs. It supports chains and agents.
Metadata: {'source': 'langchain_intro.pdf', 'page': 1}
------------------------------------------------------------
🔹 Chunk 3
Text: Retrieval-Augmented Generation improves accuracy.
Metadata: {'source': 'rag_guide.pdf', 'page': 3}
------------------------------------------------------------
🔹 Chunk 4
Text: accuracy. LangChain supports RAG pipelines.
Metadata: {'source': 'rag_guide.pdf', 'page': 3}
------------------------------------------------------------


🔹 9. Adaptive Chunking (Token-Aware NLP Pipelines)
Use token-aware methods that dynamically adjust chunk size based on:

Token count

Sentence/paragraph boundaries

📌 Prevents cutoff issues while maximizing token budget.

Instead of splitting text purely by character length or sentence count, Adaptive Chunking uses tokenizers (e.g., OpenAI tokenizer or HuggingFace tokenizer) to:

Chunk text based on token count (e.g., 200 tokens per chunk)

Avoid splitting mid-sentence/word

Align chunks with model limits for efficiency

In [20]:
from langchain.text_splitter import TokenTextSplitter

# Sample text
text = """
LangChain is a powerful framework for building applications with LLMs.
It provides support for chains, agents, tools, memory, and retrieval systems.
Token-aware chunking is crucial for efficient input to LLMs.
This method helps avoid splitting in the middle of important concepts or sentences.
"""

# Initialize token-aware splitter
text_splitter = TokenTextSplitter(
    chunk_size=30,       # tokens per chunk
    chunk_overlap=10     # overlapping tokens
)

# Perform the split
chunks = text_splitter.split_text(text)

# Show results
for i, chunk in enumerate(chunks):
    print(f"🔹 Chunk {i+1}:\n{chunk}\n{'-' * 60}")


🔹 Chunk 1:

LangChain is a powerful framework for building applications with LLMs.
It provides support for chains, agents, tools, memory, and retrieval
------------------------------------------------------------
🔹 Chunk 2:
 chains, agents, tools, memory, and retrieval systems.
Token-aware chunking is crucial for efficient input to LLMs.
This method
------------------------------------------------------------
🔹 Chunk 3:
 for efficient input to LLMs.
This method helps avoid splitting in the middle of important concepts or sentences.

------------------------------------------------------------
